<a href="https://colab.research.google.com/github/Optimus0205/Computer-Vision/blob/main/25_Training_Lane_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Jul 23 17:42:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
HOME=os.getcwd()
print(HOME)

/content


In [ ]:
%pip install ultralytics supervision roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.0/112.6 GB disk)


In [ ]:
from ultralytics import YOLO
from PIL import Image
import requests

# Download a sample image as '5.jpg' if it doesn't exist
image_path = '5.jpg'
if not requests.get('https://ultralytics.com/images/bus.jpg').ok:
    print(f"Warning: Could not download sample image from 'https://ultralytics.com/images/bus.jpg'. Please ensure a '5.jpg' file exists or provide a valid URL.")
else:
    with open(image_path, 'wb') as f:
        f.write(requests.get('https://ultralytics.com/images/bus.jpg').content)

model=YOLO('yolo11n-seg.pt')
image=Image.open(image_path)
result=model.predict(image, conf=0.25)[0]


0: 640x480 4 persons, 1 bus, 1 stop sign, 65.3ms
Speed: 6.9ms preprocess, 65.3ms inference, 79.1ms postprocess per image at shape (1, 3, 640, 480)


In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets

from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY=userdata.get('ROBOFLOW_API_KEY')
rf=Roboflow(api_key=ROBOFLOW_API_KEY)

project=rf.workspace("aditya-choudhary-ehv9p").project("l-s-kvbur")
version=project.version(1)
dataset=version.download("yolov11")

In [ ]:
dataset.location

In [ ]:
%cd {HOME}

!yolo task=segment mode=train model=yolo11m-seg.pt data={dataset.location}/data.yaml epochs=20 imgsz=640 plots=True

In [ ]:
!yolo segment predict model=yolo11n-seg.pt source='/content/5.jpg'     # predict with official model

In [ ]:
!yolo segment predict model=/content/runs/segment/train2/weights/best.pt source='/content/38244.jpg'

In [ ]:
!yolo detect predict model=yolo11n.pt source='/content/38235.jpg'